In [59]:
# here we are implementing persistence concept of langgraph

In [60]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # saves data in memory..

In [61]:
load_dotenv()

True

In [62]:
model = ChatGroq(model="llama-3.3-70b-versatile")

# 1. Implementing Persistence

In [63]:
# state
class JokeState(TypedDict):

    topic: str
    joke: str
    explaination: str

In [64]:
# joke node
def generate_joke(state: JokeState):

    prompt = f"Generate a joke on topic: {state["topic"]}"

    joke = model.invoke(prompt).content

    return {"joke": joke}

In [65]:
# explaination node
def generate_explaination(state: JokeState):

    prompt = f"generate explaination of following joke \n\n{state["joke"]}"

    explaination = model.invoke(prompt).content

    return {"explaination": explaination}

In [66]:
# create graph
graph = StateGraph(JokeState)

# create nodes
graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explaination", generate_explaination)

# create edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke","generate_explaination")
graph.add_edge("generate_explaination", END)

# create checkpointer
checkpointer = InMemorySaver()

# compile graph
workflow = graph.compile(checkpointer=checkpointer)

In [67]:
# create thread
config1 = {"configurable":{"thread_id":"101"}}

# execute graph
workflow.invoke({"topic":"pizza"},config=config1)


{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explaination': 'A delicious joke. Let\'s break it down:\n\nThe joke is a play on words, using a pun to create humor. Here\'s how it works:\n\n1. **Setup**: The joke starts by asking why the pizza is in a bad mood, which creates curiosity and sets up the expectation for a typical reason why someone (or in this case, a pizza) might be in a bad mood.\n2. **Pun**: The punchline "Because it was feeling a little crusty" uses a wordplay on the word "crusty". In one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also mean being irritable, gruff, or having a bad temper.\n3. **Wordplay**: The joke relies on the double meaning of "crusty" to create humor. The listener is expecting a reason why the pizza is in a bad mood, and instead, they get a clever connection between the pizza\'s physical characteristic (its crust)

In [68]:
# get final state - checkpoint of final state
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explaination': 'A delicious joke. Let\'s break it down:\n\nThe joke is a play on words, using a pun to create humor. Here\'s how it works:\n\n1. **Setup**: The joke starts by asking why the pizza is in a bad mood, which creates curiosity and sets up the expectation for a typical reason why someone (or in this case, a pizza) might be in a bad mood.\n2. **Pun**: The punchline "Because it was feeling a little crusty" uses a wordplay on the word "crusty". In one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also mean being irritable, gruff, or having a bad temper.\n3. **Wordplay**: The joke relies on the double meaning of "crusty" to create humor. The listener is expecting a reason why the pizza is in a bad mood, and instead, they get a clever connection between the pizza\'s physical charact

In [69]:
# get all state - checkpoints of all states
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explaination': 'A delicious joke. Let\'s break it down:\n\nThe joke is a play on words, using a pun to create humor. Here\'s how it works:\n\n1. **Setup**: The joke starts by asking why the pizza is in a bad mood, which creates curiosity and sets up the expectation for a typical reason why someone (or in this case, a pizza) might be in a bad mood.\n2. **Pun**: The punchline "Because it was feeling a little crusty" uses a wordplay on the word "crusty". In one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and golden brown. However, "crusty" can also mean being irritable, gruff, or having a bad temper.\n3. **Wordplay**: The joke relies on the double meaning of "crusty" to create humor. The listener is expecting a reason why the pizza is in a bad mood, and instead, they get a clever connection between the pizza\'s physical charac

In [70]:
# lets create another thread for joke on another topic
config2 = {"configurable":{"thread_id":"102"}}

workflow.invoke({"topic":"pasta"},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.',
 'explaination': 'A clever play on words. Here\'s a breakdown of the joke:\n\n**The Setup**: The joke starts by asking why the spaghetti refused to get married. This sets up an expectation that the punchline will be a typical reason why someone (or in this case, a food item) might not want to get married, such as fear of commitment or lack of love.\n\n**The Punchline**: The surprise comes when the reason is revealed to be "because it was afraid of getting tangled up in a lifelong commitment." This is a clever double entendre:\n\n* **Literal Meaning**: Spaghetti is a long, thin, and flexible food item that can easily become tangled or knotted. So, in a literal sense, the spaghetti might be afraid of getting physically tangled up.\n* **Figurative Meaning**: The phrase "tangled up" is also an idiomatic expression that means to become deeply i

In [71]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.', 'explaination': 'A clever play on words. Here\'s a breakdown of the joke:\n\n**The Setup**: The joke starts by asking why the spaghetti refused to get married. This sets up an expectation that the punchline will be a typical reason why someone (or in this case, a food item) might not want to get married, such as fear of commitment or lack of love.\n\n**The Punchline**: The surprise comes when the reason is revealed to be "because it was afraid of getting tangled up in a lifelong commitment." This is a clever double entendre:\n\n* **Literal Meaning**: Spaghetti is a long, thin, and flexible food item that can easily become tangled or knotted. So, in a literal sense, the spaghetti might be afraid of getting physically tangled up.\n* **Figurative Meaning**: The phrase "tangled up" is also an idiomatic expression that mean

# 2. Fault Tolerance

In [72]:
# mannual keyboard interupt and resuming the flow from last saved checkpoints - showcasing fault tolerance

In [73]:
# state
class StepState(TypedDict):

    input: str
    step1: str
    step2: str
    step3: str

In [74]:
# nodes
import time

def step_1(state: StepState) -> StepState:

    print("✅ Step 1: executed..")
    return {"step1":"done","input":state["input"]}

def step_2(state: StepState) -> StepState:

    print("⏳ Step 2: Lagging... now mannually interupt the keyboard....")
    time.sleep(30) # 30 sec. delay...
    return {"step2":"done"}

def step_3(state: StepState) -> StepState:

    print("✅ Step 3: executed..")
    return {"step3":"done"}

In [75]:
graph = StateGraph(StepState)

graph.add_node("step_1",step_1)
graph.add_node("step_2",step_2)
graph.add_node("step_3",step_3)

graph.add_edge(START, "step_1")
graph.add_edge("step_1","step_2")
graph.add_edge("step_2","step_3")
graph.add_edge("step_3",END)

checkpointer = InMemorySaver()
workflow2 = graph.compile(checkpointer=checkpointer)

In [77]:
try:
    print("▶️ Running graph.. please mannually interupt the flow at step 2....")
    workflow2.invoke({"input":"start"},config={"configurable":{"thread_id":"1"}})
except KeyboardInterrupt:
    print("❌ kernal mannully interruped... (crash simulated)")

▶️ Running graph.. please mannually interupt the flow at step 2....
✅ Step 1: executed..
⏳ Step 2: Lagging... now mannually interupt the keyboard....
❌ kernal mannully interruped... (crash simulated)


In [78]:
workflow2.get_state(config={"configurable":{"thread_id":"1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d3c-6fa8-8001-efefc310628f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-30T10:06:10.176776+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d37-603d-8000-953a31958fa6'}}, tasks=(PregelTask(id='4ffab62f-957e-964b-ecb1-6836a657dbb1', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [79]:
list(workflow2.get_state_history(config={"configurable":{"thread_id":"1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d3c-6fa8-8001-efefc310628f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-30T10:06:10.176776+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d37-603d-8000-953a31958fa6'}}, tasks=(PregelTask(id='4ffab62f-957e-964b-ecb1-6836a657dbb1', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d37-603d-8000-953a31958fa6'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-30T10:06:10.174327+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d2f

In [80]:
# now resuming from step 2 -> fault tolerance ie., we are not starting from step1 again which is aleready saved in checkpoints..
workflow2.invoke(None, config={"configurable":{"thread_id":"1"}})

⏳ Step 2: Lagging... now mannually interupt the keyboard....
✅ Step 3: executed..


{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}

In [81]:
workflow2.get_state(config={"configurable":{"thread_id":"1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe7-24f4-6099-8003-76b17c7d941a'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-30T10:07:18.094632+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe7-24ef-684a-8002-b6d154a2edb8'}}, tasks=(), interrupts=())

In [82]:
list(workflow2.get_state_history(config={"configurable":{"thread_id":"1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe7-24f4-6099-8003-76b17c7d941a'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-30T10:07:18.094632+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe7-24ef-684a-8002-b6d154a2edb8'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe7-24ef-684a-8002-b6d154a2edb8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-30T10:07:18.092770+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-9d3c-6fa8-8001-efefc310628f'}}, tasks=(PregelTask(id='6e789dcb-d8f7-bccf-9dbb-ab6af3b31419', name='s

# Time Travel

### Re-executing flow 

In [ ]:
# here we are using previous pasta joke flow. where topic is already provided and we are time travelling from joke creation...
# extract the exact flow point using checkpoint_id.
workflow.get_state(config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff8-65cf-8000-a5ec9378c7f5'}})

# in output you can see we are at exact point where topic is given and now we have to pass it to LLM for further execution.

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff8-65cf-8000-a5ec9378c7f5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-30T10:06:00.396922+00:00', parent_config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff2-63ca-bfff-8df50dc27d2b'}}, tasks=(PregelTask(id='9609bdf3-e992-5d5a-c71f-57b630f41f02', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a lifelong commitment.'}),), interrupts=())

In [ ]:
workflow.invoke(None,config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff8-65cf-8000-a5ec9378c7f5'}})

# you can see both jokes are lil bit different..

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explaination': 'Let\'s break down this joke and explore its various components to understand why it\'s funny.\n\n**The Setup**: The joke begins with a question about why spaghetti refused to get married. This is an unexpected and unusual scenario, as spaghetti is an inanimate object, a type of pasta, and it\'s not common to attribute human-like behaviors or emotions to food items. This unexpected twist already piques the listener\'s curiosity.\n\n**The Punchline**: The joke\'s humor lies in its punchline, "Because it was afraid of getting tangled up in a relationship." Here, the word "tangled" has a double meaning:\n1. **Literal Meaning**: Spaghetti is known for its long, thin, and often tangled strands. When cooked, it can easily become knotted or twisted, making it difficult to separate.\n2. **Figurative Meaning**: In relationships, "tangled 

### State Upadating

In [90]:
# we can also update sate values during time travel..
# here we are replacing value of {"topic":"pasta"} to {"topic":"samosa"}

workflow.update_state({'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff8-65cf-8000-a5ec9378c7f5'}},{'topic':'samosa'})

{'configurable': {'thread_id': '102',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18bffe-3984-60eb-8001-5645676a4ec9'}}

In [92]:
list(workflow.get_state_history(config={'configurable': {'thread_id': '102'}}))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bffe-3984-60eb-8001-5645676a4ec9'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-30T10:17:37.652318+00:00', parent_config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bfe4-3ff8-65cf-8000-a5ec9378c7f5'}}, tasks=(PregelTask(id='ba40a147-3baf-844e-4e68-aa78b0cb6213', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.', 'explaination': 'Let\'s break down this joke and explore its various components to understand why it\'s funny.\n\n**The Setup**: The joke begins with a question about why spaghetti refused to get married. This is 

In [94]:
# now we have to pass checkpoint_id of new topic's state...
workflow.invoke(None,config={'configurable': {'thread_id': '102', 'checkpoint_ns': '', 'checkpoint_id': '1f18bffe-3984-60eb-8001-5645676a4ec9'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crusty" and had a lot of "filling" emotional issues to work through!',
 'explaination': 'A deliciously clever joke. Let\'s break it down:\n\n**The Setup:** The joke starts by asking why a samosa (a popular Indian snack) went to therapy. This is an unexpected and humorous premise, as we don\'t typically associate food items with seeking psychological help.\n\n**The Punchline:** The joke relies on a play on words, using the physical characteristics of a samosa to make a pun on emotional states. Here\'s how it works:\n\n* "Feeling a little \'crusty\'": Samosas are typically crispy on the outside, with a crunchy crust. In this joke, "crusty" is used to describe the samosa\'s emotional state, implying that it\'s feeling a bit rough, hardened, or defensive. This is a clever wordplay, as "crusty" can also mean being irritable or gruff.\n* "Had a lot of \'filling\' emotional issues to work throu

In [95]:
list(workflow.get_state_history(config={"configurable":{"thread_id":"102"}}))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crusty" and had a lot of "filling" emotional issues to work through!', 'explaination': 'A deliciously clever joke. Let\'s break it down:\n\n**The Setup:** The joke starts by asking why a samosa (a popular Indian snack) went to therapy. This is an unexpected and humorous premise, as we don\'t typically associate food items with seeking psychological help.\n\n**The Punchline:** The joke relies on a play on words, using the physical characteristics of a samosa to make a pun on emotional states. Here\'s how it works:\n\n* "Feeling a little \'crusty\'": Samosas are typically crispy on the outside, with a crunchy crust. In this joke, "crusty" is used to describe the samosa\'s emotional state, implying that it\'s feeling a bit rough, hardened, or defensive. This is a clever wordplay, as "crusty" can also mean being irritable or gruff.\n* "Had a lot of \'filling\' emotional 